# 05 — Gold: Métricas Analíticas
## Módulo 3 | Camada Gold | Seção do Exame: S3 (22%)

**O que este notebook faz:**
- Cria `gold.daily_summary` — resumo diário por cidade (avg/min/max temp, umidade, vento)
- Cria `gold.city_ranking_temp` — ranking de cidades por temperatura atual
- Cria `gold.weather_alerts` — view com alertas: HIGH_TEMP, RAIN_RISK, STORM_RISK
- Aplica `OPTIMIZE` + `ZORDER` nas tabelas Gold

**Princípio Gold:** Dados prontos para consumo direto por BI, dashboards e ML.  
Toda lógica de negócio fica aqui — nenhuma transformação nos dashboards.

**Conceito do exame (S3):** Diferença entre `TABLE`, `VIEW` e `MATERIALIZED VIEW`:  
- `TABLE` Delta = dados físicos persistidos, alta performance de leitura  
- `VIEW` = consulta lógica, sempre atual, sem armazenamento físico  
- `MATERIALIZED VIEW` (Lakeflow) = resultado pré-computado e atualizado automaticamente

---

### ✅ Checklist deste notebook
- [ ] Célula 1 — `daily_summary` criada com `avg_temp`, `min_temp`, `max_temp`, `avg_humidity`
- [ ] Célula 2 — `city_ranking_temp` com coluna `temp_rank`
- [ ] Célula 3 — View `weather_alerts` criada com coluna `alert_type`
- [ ] Célula 4 — OPTIMIZE + ZORDER executados
- [ ] Validei as 3 tabelas/views no Data Explorer

In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")
CATALOG = dbutils.widgets.get("catalog")

print(f"Catálogo : {CATALOG}")
print(f"Silver   : {CATALOG}.silver")
print(f"Gold     : {CATALOG}.gold")

---
## Célula 1 — `gold.daily_summary`

Agrega as previsões da Silver por cidade e por dia,  
calculando médias, extremos e a condição meteorológica dominante.

In [ ]:
%sql
CREATE OR REPLACE TABLE workshop_weather.gold.daily_summary
USING DELTA
COMMENT 'Resumo diário por cidade: temperatura, umidade, vento e condição dominante'
AS
SELECT
    city_id,
    city_name,
    country,
    CAST(DATE_TRUNC('DAY', forecast_timestamp) AS DATE)  AS forecast_date,
    ROUND(AVG(temp), 2)                                  AS avg_temp,
    ROUND(MIN(temp), 2)                                  AS min_temp,
    ROUND(MAX(temp), 2)                                  AS max_temp,
    ROUND(AVG(humidity), 1)                              AS avg_humidity,
    ROUND(AVG(pop) * 100, 1)                             AS avg_rain_prob_pct,
    -- Condição dominante do dia (valor mais frequente)
    MODE(weather_main)                                   AS dominant_weather,
    COUNT(*)                                             AS slot_count,
    CURRENT_TIMESTAMP()                                  AS _updated_at
FROM workshop_weather.silver.forecast
WHERE _dq_status = 'valid'
GROUP BY city_id, city_name, country, DATE_TRUNC('DAY', forecast_timestamp)
ORDER BY city_name, forecast_date;

In [ ]:
%sql
SELECT * FROM workshop_weather.gold.daily_summary
ORDER BY city_name, forecast_date
LIMIT 20;

---
## Célula 2 — `gold.city_ranking_temp`

Ranking de cidades pela temperatura **atual** (mais recente na Silver),  
usando `ROW_NUMBER()` window function particionada por país.

In [ ]:
%sql
CREATE OR REPLACE TABLE workshop_weather.gold.city_ranking_temp
USING DELTA
COMMENT 'Ranking de cidades por temperatura atual, particionado por país'
AS
WITH latest AS (
    -- Pega apenas o registro mais recente por cidade
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY city_id
            ORDER BY weather_timestamp DESC
        ) AS rn
    FROM workshop_weather.silver.current_weather
    WHERE _dq_status = 'valid'
)
SELECT
    city_id,
    city_name,
    country,
    lat,
    lon,
    ROUND(temp, 1)        AS temp,
    ROUND(feels_like, 1)  AS feels_like,
    humidity,
    weather_main,
    weather_timestamp,
    -- Ranking dentro do país: 1 = cidade mais quente
    ROW_NUMBER() OVER (
        PARTITION BY country
        ORDER BY temp DESC
    )                     AS temp_rank,
    CURRENT_TIMESTAMP()   AS _updated_at
FROM latest
WHERE rn = 1;

In [ ]:
%sql
SELECT city_name, country, temp, feels_like, weather_main, temp_rank
FROM workshop_weather.gold.city_ranking_temp
ORDER BY temp_rank;

---
## Célula 3 — View `gold.weather_alerts`

**View vs Table:** Esta é uma `VIEW` lógica — sempre reflete os dados atuais da Silver,  
sem armazenamento físico. Ideal para alertas em tempo real.

**Regras de alerta:**
- `HIGH_TEMP` → temperatura prevista > 35°C
- `RAIN_RISK` → probabilidade de precipitação > 70%
- `STORM_RISK` → condição `Thunderstorm` na previsão

In [ ]:
%sql
CREATE OR REPLACE VIEW workshop_weather.gold.weather_alerts AS
SELECT
    city_id,
    city_name,
    country,
    forecast_timestamp,
    ROUND(temp, 1)           AS temp,
    ROUND(pop * 100, 0)      AS rain_prob_pct,
    weather_main,
    weather_description,
    -- Tipo de alerta: lógica de negócio centralizada aqui, não no dashboard
    CASE
        WHEN weather_main = 'Thunderstorm'  THEN 'STORM_RISK'
        WHEN temp > 35                      THEN 'HIGH_TEMP'
        WHEN pop > 0.70                     THEN 'RAIN_RISK'
        ELSE                                     'NORMAL'
    END                      AS alert_type,
    -- Nível de urgência para ordenação em dashboards
    CASE
        WHEN weather_main = 'Thunderstorm'  THEN 1
        WHEN temp > 35                      THEN 2
        WHEN pop > 0.70                     THEN 3
        ELSE                                     4
    END                      AS alert_priority
FROM workshop_weather.silver.forecast
WHERE _dq_status = 'valid'
  AND forecast_timestamp >= CURRENT_TIMESTAMP()  -- apenas previsões futuras
;

In [ ]:
%sql
-- Resumo de alertas por cidade
SELECT
    city_name,
    alert_type,
    COUNT(*) AS ocorrencias,
    MIN(forecast_timestamp) AS primeiro_alerta
FROM workshop_weather.gold.weather_alerts
WHERE alert_type != 'NORMAL'
GROUP BY city_name, alert_type
ORDER BY city_name, alert_type;

---
## Célula 4 — OPTIMIZE + ZORDER nas tabelas Gold

**Por que OPTIMIZE nas tabelas Gold?**
- Gold é a camada mais consultada (BI, dashboards, ML)
- `OPTIMIZE` compacta muitos arquivos pequenos em poucos arquivos grandes
- `ZORDER BY` co-localiza dados por colunas de filtro frequente → menos arquivos lidos por query

**Resultado esperado:** `DESCRIBE DETAIL` antes/depois mostra redução no número de arquivos.

In [ ]:
%sql
-- Inspeciona antes do OPTIMIZE
SELECT name, numFiles, sizeInBytes
FROM (
    DESCRIBE DETAIL workshop_weather.gold.daily_summary
);

In [ ]:
%sql
-- Compacta arquivos e ordena por cidade + data (colunas mais usadas em filtros)
OPTIMIZE workshop_weather.gold.daily_summary
ZORDER BY (city_id, forecast_date);

In [ ]:
%sql
OPTIMIZE workshop_weather.gold.city_ranking_temp
ZORDER BY (country, temp_rank);

---
## ✅ Checklist Final — Módulo 3 concluído?

- [ ] `daily_summary` tem linhas com `avg_temp`, `min_temp`, `max_temp` por cidade/dia
- [ ] `city_ranking_temp` tem coluna `temp_rank` (1 = mais quente no país)
- [ ] `weather_alerts` view retorna previsões com `alert_type`
- [ ] OPTIMIZE executado — `numFiles` reduziu no `DESCRIBE DETAIL`
- [ ] Você entende quando usar TABLE vs VIEW vs MATERIALIZED VIEW

**Próximo: `06_delta_advanced_features.ipynb`**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*